# IC 4040 – PanSTARRS Downloader

<div class="alert alert-block alert-info">
<b>Environment:</b> Run this notebook in the <code>stenv</code> conda environment.
</div>

## Imports

In [ ]:
# Python Imports
import os
from pathlib import Path

# Astropy Collaboration Imports
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import QTable, Table
from astroquery.vizier import Vizier

# 3rd Party Imports
import numpy as np


## Notebook Setup

In [ ]:
if Path.cwd().name != 'PANSTARRS':
    if Path.cwd().name == 'Notebooks':
        os.chdir('../Data/PANSTARRS')
    else:
        raise RuntimeError(
            'This notebook must be run from the PANSTARRS directory.'
        )
print(f'Current Directory: {Path.cwd()}')

In [ ]:
# Input data
NED_DATA_FILE = Path('../NED/IC4040-NED_Data.ecsv')

# Query parameters
FOV_RADIUS_ARCMIN = 20.0     # arcminutes -- match image FOV
MIN_DETECTIONS = 4           # minimum PS1 detections for reliability
STAR_GAL_SEP_THRESH = 0.05   # rmag - rKmag (VizieR II/349 columns) < threshold -> stellar
GAIA_MATCH_TOL_ARCSEC = 1.0  # positional cross-match tolerance (arcsec)

# Individual output file paths
OUT_CATALOG_ECSV = Path('IC4040-PANSTARRS-AlignmentStars.ecsv')
OUT_COORDS_ECSV = Path('IC4040-PANSTARRS-AlignmentStars-Coordinates.ecsv')
OUT_COORDS_REG = Path('IC4040-PANSTARRS-AlignmentStars-Coordinates.reg')
OUT_TWEAKREG_CAT = Path('IC4040-PANSTARRS-RefCatalog-icrs.txt')


## Load External Data

Load the NED galaxy data table to obtain the galaxy's sky coordinates,
which define the centre of the PanSTARRS search cone.

In [ ]:
ned_data_table = Table.read(NED_DATA_FILE)
gal_crd = SkyCoord(
    ra=ned_data_table['RA'][0],
    dec=ned_data_table['DEC'][0],
    unit='deg',
    frame='fk5'
)
print(f'Galaxy Coordinates (FK5):  {gal_crd}')
print(f'Galaxy Coordinates (ICRS): {gal_crd.icrs}')

## Query PanSTARRS

Query the PanSTARRS PS1 DR1 mean-object catalogue for all sources within
`FOV_RADIUS_ARCMIN` arcminutes of the galaxy centre using
`astroquery.vizier.Vizier` (VizieR catalog `II/349/ps1`).  DR1 positions
are sufficient for TweakReg astrometric alignment.  The full result (all
source types) is returned here; stellar filtering is applied in the next
section.


In [ ]:
print('Querying PanSTARRS PS1 DR1 via VizieR (II/349) ...')
_vizier = Vizier(
    catalog='II/349/ps1',
    columns=['RAJ2000', 'DEJ2000', 'Nd', 'rmag', 'rKmag'],
    row_limit=-1,
)
_result = _vizier.query_region(
    gal_crd.icrs,
    radius=FOV_RADIUS_ARCMIN * u.arcmin,
)
ps1_table = _result[0]
print(f'Found {len(ps1_table):d} total PS1 objects in the FOV')


## Filter to Stellar Sources

Restrict the PanSTARRS catalogue to point-like (stellar) sources suitable
for TweakReg alignment using two quality criteria:

1. **Minimum detections** — require at least `MIN_DETECTIONS` individual
   PS1 epoch detections to exclude spurious or poorly measured objects.
2. **PSF – Kron magnitude cut** — for stellar sources the PSF and Kron
   magnitudes are nearly identical; sources with
   `rmag − rKmag < STAR_GAL_SEP_THRESH` are classified as point-like.
   Extended sources (galaxies) have significantly larger Kron magnitudes,
   giving a positive difference that exceeds the threshold.


In [ ]:
# Extract r-band PSF and Kron magnitudes as plain float arrays.
# VizieR returns masked columns; fill masked entries with NaN.
r_psf = np.array(ps1_table['rmag'].filled(np.nan), dtype=float)
r_kron = np.array(ps1_table['rKmag'].filled(np.nan), dtype=float)

# Boolean masks
det_mask = np.array(ps1_table['Nd'].filled(0), dtype=int) >= MIN_DETECTIONS
valid_r_mask = np.isfinite(r_psf) & np.isfinite(r_kron)
star_mask = (r_psf - r_kron) < STAR_GAL_SEP_THRESH
valid_mask = det_mask & valid_r_mask & star_mask

ps1_objects = ps1_table[valid_mask]

print(f'Total PS1 objects queried   : {len(ps1_table):d}')
print(f'After detection cut         : {int(det_mask.sum()):d}')
print(f'After valid r-band cut      : {int((det_mask & valid_r_mask).sum()):d}')
print(f'After PSF-Kron stellar cut  : {int(valid_mask.sum()):d}')

# Build SkyCoord array for the filtered stellar sources
ps1_coords = SkyCoord(
    ra=ps1_objects['RAJ2000'],
    dec=ps1_objects['DEJ2000'],
    unit='deg',
    frame='icrs'
)
print(f'Stellar coordinate array built: {len(ps1_coords):d} entries')


## Write Outputs

Write the filtered stellar catalogue to four output files:

1. **ECSV catalogue** — full `ps1_objects` table with PanSTARRS metadata.
2. **Coordinates ECSV** — `SkyCoord` column only, for convenient re-loading.
3. **DS9 region file** — point markers for visual inspection.
4. **TweakReg plain-text catalogue** — whitespace-separated RA Dec (ICRS
   decimal degrees) as required by `TweakReg`.

In [ ]:
# 1. Write the full filtered table
ps1_objects.write(OUT_CATALOG_ECSV, overwrite=True)
print(f'Wrote full catalogue  -> {OUT_CATALOG_ECSV}')

In [ ]:
# 2. Write SkyCoord-only ECSV
# Reload with: QTable.read(OUT_COORDS_ECSV)['SkyCoord']
QTable([ps1_coords], names=['SkyCoord']).write(
    OUT_COORDS_ECSV, overwrite=True
)
print(f'Wrote coordinates     -> {OUT_COORDS_ECSV}')

In [ ]:
# 3. Write DS9 region file with cyan 'cross' markers (all PS1-only sources)
with open(OUT_COORDS_REG, 'w') as fid:
    fid.write('# Region file format: DS9 version 4.1\n')
    fid.write('global color=cyan\n')
    fid.write('icrs\n')
    for crd in ps1_coords:
        fid.write(
            f'point({crd.ra.deg:.8f},{crd.dec.deg:.8f}) # point=cross\n'
        )

print(f'Wrote DS9 regions     -> {OUT_COORDS_REG}')


In [ ]:
# 4. Write TweakReg plain-text catalogue (whitespace-separated RA Dec)
with open(OUT_TWEAKREG_CAT, 'w') as fid:
    for crd in ps1_coords:
        fid.write(
            f'{crd.icrs.ra.value:<20.8f} {crd.icrs.dec.value:<20.8f}\n'
        )

print(f'Wrote TweakReg cat.   -> {OUT_TWEAKREG_CAT}')
print(f'Total alignment stars: {len(ps1_coords):d}')

## Combined GAIA + PanSTARRS Catalogue

Cross-match the PanSTARRS stellar catalogue against the pre-existing GAIA
catalogue using a positional tolerance of `GAIA_MATCH_TOL_ARCSEC` arcsec.
Sources with a GAIA counterpart retain the higher-precision GAIA position;
PanSTARRS-only sources (no GAIA match within the tolerance) are appended to
the GAIA list.  The same four output formats are written as for the
individual catalogue.

In [ ]:
# GAIA input (full table for coordinates)
GAIA_TABLE_FILE = Path('../GAIA/IC4040-GAIA-AlignmentStars.ecsv')

# Combined output file paths
OUT_COMB_DIR = Path('Combined')
OUT_COMB_CATALOG_ECSV = (
    OUT_COMB_DIR / 'IC4040-PS1-GAIA-Combined-AlignmentStars.ecsv'
)
OUT_COMB_COORDS_ECSV = OUT_COMB_DIR / (
    'IC4040-PS1-GAIA-Combined-AlignmentStars-Coordinates.ecsv'
)
OUT_COMB_COORDS_REG = OUT_COMB_DIR / (
    'IC4040-PS1-GAIA-Combined-AlignmentStars-Coordinates.reg'
)
OUT_COMB_TWEAKREG_CAT = (
    OUT_COMB_DIR / 'IC4040-PS1-GAIA-Combined-RefCatalog-icrs.txt'
)

In [ ]:
# Make the Combined directory
OUT_COMB_DIR.mkdir(exist_ok=True)

### Cross-Match PanSTARRS Against GAIA

Load the GAIA alignment-star catalogue and positionally cross-match each
PanSTARRS stellar source against the GAIA positions.  For each PS1 star
the nearest GAIA neighbour is found; sources separated by more than
`GAIA_MATCH_TOL_ARCSEC` arcsec from every GAIA source are classified as
PanSTARRS-only and appended to the combined list.

In [ ]:
# Load GAIA table
gaia_full_table = Table.read(GAIA_TABLE_FILE)
gaia_coords = SkyCoord(
    ra=gaia_full_table['ra'],
    dec=gaia_full_table['dec'],
    unit='deg',
    frame='icrs'
)
print(f'Loaded {len(gaia_coords):d} GAIA sources')

# Positional cross-match: for each PS1 star find the nearest GAIA source
print(f'Cross-matching {len(ps1_coords):d} PS1 stars against GAIA ...')
idx, sep2d, _ = ps1_coords.match_to_catalog_sky(gaia_coords)
ps1_only_mask = sep2d > (GAIA_MATCH_TOL_ARCSEC * u.arcsec)
ps1_only_coords = ps1_coords[ps1_only_mask]
n_ps1_only = int(ps1_only_mask.sum())
n_matched = int((~ps1_only_mask).sum())

tol_str = f'{GAIA_MATCH_TOL_ARCSEC:.1f}"'
print(f'PS1 stars with GAIA match (< {tol_str}): {n_matched:d}')
print(f'PS1-only sources (no GAIA match)       : {n_ps1_only:d}')
print(f'Total combined sources                 : {len(gaia_coords) + n_ps1_only:d}')

# Build combined SkyCoord: all GAIA positions first, then PS1-only
combined_coords = SkyCoord(
    ra=[*gaia_coords.ra.deg, *ps1_only_coords.ra.deg],
    dec=[*gaia_coords.dec.deg, *ps1_only_coords.dec.deg],
    unit='deg',
    frame='icrs'
)
source_labels = ['GAIA'] * len(gaia_coords) + ['PS1'] * n_ps1_only

### Write Combined Outputs

Write the combined catalogue to the same four formats used for the
individual PanSTARRS catalogue.  The full ECSV catalogue includes a
`source` column (`'GAIA'` or `'PS1'`) indicating the origin of each
position.

In [ ]:
# 1. Write the combined catalogue with ra, dec, and source flag
comb_catalog = QTable(
    [combined_coords.ra, combined_coords.dec, source_labels],
    names=['ra', 'dec', 'source']
)
comb_catalog.write(OUT_COMB_CATALOG_ECSV, overwrite=True)
print(f'Wrote combined catalogue  -> {OUT_COMB_CATALOG_ECSV}')

In [ ]:
# 2. Write SkyCoord-only ECSV
# Reload with: QTable.read(OUT_COMB_COORDS_ECSV)['SkyCoord']
QTable([combined_coords], names=['SkyCoord']).write(
    OUT_COMB_COORDS_ECSV, overwrite=True
)
print(f'Wrote coordinates         -> {OUT_COMB_COORDS_ECSV}')

In [ ]:
# 3. Write DS9 region file – GAIA sources: 'x' marker, PS1-only: 'cross' marker
with open(OUT_COMB_COORDS_REG, 'w') as fid:
    fid.write('# Region file format: DS9 version 4.1\n')
    fid.write('global point=x color=cyan\n')
    fid.write('icrs\n')
    for crd, lbl in zip(combined_coords, source_labels):
        end = ' # point=cross color=green' if lbl == 'PS1' else ''
        fid.write(
            f'point({crd.ra.deg:.8f},{crd.dec.deg:.8f}){end}\n'
        )

print(f'Wrote DS9 regions         -> {OUT_COMB_COORDS_REG}')


In [ ]:
# 4. Write TweakReg plain-text catalogue (whitespace-separated RA Dec)
with open(OUT_COMB_TWEAKREG_CAT, 'w') as fid:
    for crd in combined_coords:
        fid.write(
            f'{crd.icrs.ra.value:<20.8f} {crd.icrs.dec.value:<20.8f}\n'
        )

print(f'Wrote TweakReg cat.       -> {OUT_COMB_TWEAKREG_CAT}')
print(f'Total combined stars:      {len(combined_coords):d}')